# **ViT(Vision Transformer)**

## 1.환경준비

### (1)라이브러리 로딩

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, ViTForImageClassification

import os
import numpy as np
from keras.datasets import cifar10, fashion_mnist
from sklearn.metrics import *
from tqdm.auto import tqdm

# 공통: 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### (2) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (3) OpenAI API Key 등록
* 환경변수로 key 등록

In [ ]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/langchain/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

* ⚠️ 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
print(os.environ['OPENAI_API_KEY'][:40])

## 2.ViT 사용해보기

### (1) ViT 기본 모델 다운로드

In [ ]:
# 모델 다운로드
model_id = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_id)
model1 = ViTForImageClassification.from_pretrained(model_id).to(device)

### (2) 모델 사용

* 이미지 예측 함수 생성
    * .convert("RGB") : 어떤 포맷의 이미지든 명시적으로 3채널 RGB로 변환되므로 안정적으로 처리 가능.

In [ ]:
def predict_single_image(image_path: str):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model1(**inputs).logits
    pred = int(logits.argmax(-1))
    pred_label = model1.config.id2label.get(pred, str(pred))
    print(f"{image_path} -> {pred_label}")

In [ ]:
from google.colab import files
uploaded = files.upload() # 로컬 PC에서 a.png 선택

In [ ]:
predict_single_image("a.png")

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 suni_bang1.jpg 선택

In [ ]:
predict_single_image("suni_bang1.jpg")

## 3.파인튜닝 모델 : Fashion-mnist

### (1) 데이터 준비

In [ ]:
# 케라스 데이터셋으로 부터 fashion_mnist 불러오기
(_, _), (x_val, y_val) = fashion_mnist.load_data()

In [ ]:
# 타깃 클래스
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot" ]

### (2) 모델 다운로드
* 허깅페이스에 올라온 Fashion-MNIST 파인튜닝된 ViT 모델 : Kankanaghosh/vit-fashion-mnist
* 모델에 맞는 전처리기(processor)를 불러옴. (리사이즈, 정규화 등 자동 처리)
* 사전 학습된 ViT 분류 모델을 다운로드해서 GPU/CPU(device)에 올림

In [ ]:
model_id = "Kankanaghosh/vit-fashion-mnist"
processor = AutoImageProcessor.from_pretrained(model_id)
model = ViTForImageClassification.from_pretrained(model_id).to(device)

### (3) 검증평가

In [ ]:
BATCH = 128   # 평가 배치 크기를 128로 설정
n = len(x_val)    # 검증 이미지 개수(n) 확인
y_pred = np.zeros(n, dtype=int)   # 예측 결과(y_pred)를 저장할 배열을 0으로 초기화

model.eval()   # 모델을 평가 모드로 전환
with torch.no_grad():    # 평가 시에는 학습(gradient 계산)이 필요 없으므로 메모리 절약을 위해 사용
    for i in tqdm(range(0, n, BATCH)):  # 0 ~ n까지 128단위로 반복
        j = min(i + BATCH, n)        # 마지막 배치에서 n을 넘지 않도록 범위 제한

        # Fashion-MNIST는 1채널 → ViT는 3채널 기대 → RGB 변환
        # 해당 배치 이미지를 PIL 이미지로 변환 후 리스트로 저장
        imgs = [Image.fromarray(x_val[k]).convert("RGB") for k in range(i, j)]

        # 이미지 전처리
        inputs = processor(images=imgs, return_tensors="pt").to(device)

        # 예측, 로짓값에 argmax 적용
        logits = model(**inputs).logits
        pred = logits.argmax(dim=-1).cpu().numpy()

        # 해당 배치 구간(i~j)에 예측값을 채워 넣음.
        y_pred[i:j] = pred


In [ ]:
print(confusion_matrix(y_val, y_pred))
print('-'*100)
print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

## 4.실습 : 파인튜닝 모델 : CIFAR10
* cifar10 데이터로 파인 튜닝된 모델에 대해 검증해 봅니다.

### (1) 데이터 준비

In [ ]:
_, (x_val, y_val) = cifar10.load_data()

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

### (2) 모델 다운로드

In [ ]:
# vit - cifar10 파인튜닝 모델
model_id = "nateraw/vit-base-patch16-224-cifar10"



### (3) 검증평가